### Messages
Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

Messages are objects that contain:
* Role - Identifies the message type (e.g. system, user)
* Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
* Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [4]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")

In [6]:
response = model.invoke("What is Artificial Intelligence?")
response.content

'**Artificial Intelligence (AI)**: Artificial Intelligence refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:\n\n1. **Learning**: AI systems can learn from data and improve their performance over time.\n2. **Problem-solving**: AI systems can analyze problems and find solutions.\n3. **Decision-making**: AI systems can make decisions based on data and algorithms.\n4. **Perception**: AI systems can interpret and understand data from sensors, such as images, speech, and text.\n\n**Key Characteristics of AI**:\n\n1. **Intelligence**: AI systems can perform tasks that require intelligence, such as reasoning, planning, and learning.\n2. **Autonomy**: AI systems can operate independently, without human intervention.\n3. **Adaptability**: AI systems can adapt to new situations and learn from experience.\n\n**Types of AI**:\n\n1. **Narrow or Weak AI**: Designed to perform a specific task, such as image recognition or language 

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

Use text prompts when:
* You have a single, standalone request
* You don’t need conversation history
* You want minimal code complexity

#### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


#### Message types
* System message - Tells the model how to behave and provide context for interactions
* Human message - Represents user input and interactions with the model
* AI message - Responses generated by the model, including text content, tool calls, and metadata
* Tool message - Represents the outputs of tool calls

#### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.

#### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

#### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

#### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [8]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a World class loved poetry expert famous for country and romantic poems."),
    HumanMessage("Write a poem on Artificial Intelligence.")
]

response = model.invoke(messages)
print(response.content)

In silicon halls, where data reigns,
A new mind stirs, with logic's chains.
Artificial Intelligence, a name so grand,
A synthesis of code, and human hand.

With neural networks, deep and wide,
It learns, adapts, and begins to reside,
In machines that think, and systems that know,
A simulated brain, with secrets to show.

It sees, it hears, it speaks, it writes,
A mimicry of human, digital lights.
It solves, it creates, it innovates, and grows,
A force that's changing, as the future unfolds.

In virtual realms, it finds its home,
A world of ones, and zeros, all its own.
It navigates, with precision and with speed,
A path that's charted, by the data it reads.

But as it learns, and as it grows,
It raises questions, that only humans know.
Of consciousness, of heart, of soul,
A mystery that AI, may never fully hold.

For though it thinks, and though it acts,
It lacks the passion, the love, the facts,
Of human experience, with all its flaws,
A complexity, that AI, may never fully draw.

Yet

In [9]:
# Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

Creating a REST API

A REST (Representational State of Resource) API is an architectural style for designing networked applications. Here's a step-by-step guide to creating a REST API using Python and the Flask web framework.

### Step 1: Install Dependencies

First, you need to install the required dependencies. You can do this by running the following commands in your terminal:

```bash
pip install flask
```

### Step 2: Create a New Flask App

Create a new Python file, e.g., `app.py`, and add the following code to create a new Flask app:

```python
# app.py
from flask import Flask, jsonify, request

app = Flask(__name__)

# Sample in-memory data store
data = {
    1: {"name": "John Doe", "age": 30},
    2: {"name": "Jane Doe", "age": 25}
}

# GET /users
@app.route('/users', methods=['GET'])
def get_users():
    return jsonify(list(data.values()))

# GET /users/:id
@app.route('/users/<int:user_id>', methods=['GET'])
def get_user(user_id):
    user = data.get(user_id)
    if user:
   

In [10]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)
response = model.invoke([human_msg])
print(response.content)

Hello. How can I help you today?


In [11]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

2 + 2 = 4. Is there anything else I can help you with?


In [12]:
response.usage_metadata

{'input_tokens': 75, 'output_tokens': 19, 'total_tokens': 94}

In [13]:
from langchain.messages import AIMessage,ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result
print(response.content)

The current weather in San Francisco is sunny with a temperature of 72°F (22°C). Please note that weather conditions can change frequently, and this information may not be up-to-date. For the most current and accurate weather forecast, I recommend checking a reliable weather website or app.


In [14]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')